# Build ROQ Basis with `mlgw_bns_jax` (JAX-native, GPU-accelerated)

This notebook builds a **Reduced Order Quadrature (ROQ)** basis for the
`mlgw_bns_jax` gravitational waveform surrogate, using our custom
JAX-native ROQ builder.

**Key features:**
- Runs on **GPU** via `jax[cuda12]` for accelerated waveform generation
- **JIT-compiles** waveform kernels on a small warm-up call before batch evaluation
- **`jax.vmap`** for batched waveform generation (larger batches on GPU)
- **Streaming memory** management — never loads all training waveforms at once
- **Checkpoint / resume** — can be restarted after interruption without losing progress

**Runtime requirements:**
- Google Colab with a **GPU runtime** (T4, A100, or L4)
- ~15 GB RAM (standard Colab is fine)
- Several hours for the full build (depends on GPU type)

---
⚠️ **Before running:** Go to `Runtime → Change runtime type → GPU`

## 1. Install dependencies

We install JAX with CUDA 12 support first (before any JAX import),
then the remaining dependencies. The order matters to avoid
JAX CPU/GPU conflicts.

In [ ]:
# Step 1: Install JAX with CUDA 12 support
# This MUST be done before importing JAX to get GPU support
!pip install --upgrade "jax[cuda12]" 2>&1 | tail -3

# Step 2: Install remaining dependencies
!pip install numpy scipy h5py 2>&1 | tail -3

print("\n✓ Dependencies installed")

In [ ]:
# Verify GPU is available BEFORE importing the ROQ builder
import os

# Force GPU platform
os.environ["JAX_PLATFORMS"] = "cuda"

import jax
jax.config.update("jax_enable_x64", True)

devices = jax.devices()
print(f"JAX version: {jax.__version__}")
print(f"Devices: {devices}")
print(f"Default backend: {jax.default_backend()}")

if jax.default_backend() != "gpu":
    print("\n⚠️  WARNING: No GPU detected!")
    print("Go to Runtime → Change runtime type → GPU")
    print("Falling back to CPU (will be much slower)...")
    os.environ["JAX_PLATFORMS"] = "cpu"
else:
    gpu = devices[0]
    print(f"\n✓ GPU detected: {gpu}")
    # Quick GPU sanity check
    import jax.numpy as jnp
    x = jnp.ones(1000)
    _ = (x @ x).block_until_ready()
    print("✓ GPU compute verified")

## 2. Clone the repository and set up

In [ ]:
import os

REPO_URL = "https://github.com/saulo-albuquerque-phys/mlgw_bns_jax.git"
BRANCH = "copilot/implement-reduced-order-quadrature"  # branch with JAX ROQ builder
REPO_DIR = "/content/mlgw_bns_jax"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    print(f"Repository already cloned at {REPO_DIR}")
    !cd {REPO_DIR} && git pull origin {BRANCH}

os.chdir(REPO_DIR)
print(f"\n✓ Working directory: {os.getcwd()}")
print(f"✓ Branch: {BRANCH}")

In [ ]:
# Verify that the waveform model file exists
import os

model_path = "mlgw_bns_jax_model.h5"
assert os.path.isfile(model_path), (
    f"Model file '{model_path}' not found. "
    "Make sure the repo contains the HDF5 model."
)
size_mb = os.path.getsize(model_path) / 1e6
print(f"✓ Model file found: {model_path} ({size_mb:.1f} MB)")

# Check config file
config_path = "config_roq_mlgw_bns_jax_gw170817.ini"
assert os.path.isfile(config_path), f"Config file '{config_path}' not found."
print(f"✓ Config file found: {config_path}")

# Check ROQ builder
assert os.path.isfile("roq_builder_jax.py"), "ROQ builder not found."
print(f"✓ ROQ builder found: roq_builder_jax.py")

## 3. Configure the ROQ build

We load the configuration and adjust GPU-specific parameters:
- **`waveform_batch_size`**: On GPU with 16 GB VRAM, we can use larger batches (32–64) since GPU has more memory bandwidth. On T4 (16 GB), use 32; on A100 (40/80 GB), use 64.
- **`projection_batch_size`**: How many waveforms to hold in RAM at once for projection-error computation.

In [ ]:
import sys
sys.path.insert(0, ".")

from roq_builder_jax import ROQConfig

cfg = ROQConfig.from_ini("config_roq_mlgw_bns_jax_gw170817.ini")

# ── GPU-optimized batch sizes ───────────────────────────────────
# On GPU we can afford larger vmap batches than on CPU.
# Adjust based on your GPU VRAM:
#   T4   (16 GB):  batch_size=32,  proj_batch=500
#   A100 (40 GB):  batch_size=64,  proj_batch=1000
#   L4   (24 GB):  batch_size=48,  proj_batch=500

if jax.default_backend() == "gpu":
    cfg.waveform_batch_size = 32
    cfg.projection_batch_size = 500
    print("Using GPU-optimized batch sizes:")
else:
    cfg.waveform_batch_size = 8
    cfg.projection_batch_size = 200
    print("Using CPU batch sizes (GPU not available):")

print(f"  waveform_batch_size   = {cfg.waveform_batch_size}")
print(f"  projection_batch_size = {cfg.projection_batch_size}")

# ── Output directory (persists to Google Drive if mounted) ──────
# Uncomment the next 3 lines to save results to Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# cfg.output_dir = '/content/drive/MyDrive/roq_basis_mlgw_bns_jax'

print(f"\nFull configuration:")
print(f"  Frequency range  : [{cfg.f_min}, {cfg.f_max}] Hz")
print(f"  Segment length   : {cfg.seglen} s")
print(f"  Frequency points : {cfg.n_freq}")
print(f"  Tolerance (lin)  : {cfg.tolerance_lin:.1e}")
print(f"  Tolerance (qua)  : {cfg.tolerance_qua:.1e}")
print(f"  Training cycles  : {cfg.training_set_sizes}")
print(f"  Output directory : {cfg.output_dir}")

## 4. JIT warm-up (compile kernels before batch work)

This is the **critical step** for performance: we compile both the
single-waveform and `vmap`-batched kernels **once** with a throwaway
evaluation.  JAX/XLA will trace and compile to GPU machine code here.
All subsequent calls reuse the cached compiled kernels — no Python overhead.

On GPU, JIT compilation typically takes 5–15 seconds.

In [ ]:
import time
import numpy as np
import jax.numpy as jnp

from roq_builder_jax import (
    _load_predictor,
    warmup_jit,
    generate_waveforms_batch,
    normalise,
    sample_parameters,
)

# Load the waveform model
print("Loading waveform model...")
predict_fn = _load_predictor(cfg.model_path)
print("✓ Model loaded")

# JIT compile: single + vmap-batched waveform kernels
print(f"\nJIT compiling kernels (batch_size={cfg.waveform_batch_size})...")
t0 = time.time()
precompiled = warmup_jit(
    predict_fn,
    cfg.frequencies,
    cfg.waveform_batch_size,
    verbose=1,
)
print(f"\n✓ Total JIT warm-up: {time.time() - t0:.1f}s")

## 5. Quick benchmark: JIT-compiled waveform speed

After JIT compilation, let's measure the actual throughput.
The first call above was slow (compilation). Now calls are fast.

In [ ]:
rng = np.random.default_rng(42)
test_params = sample_parameters(rng, cfg.waveform_batch_size * 10, cfg)

# Warm: generate a batch (already compiled)
_ = generate_waveforms_batch(
    predict_fn, test_params[:cfg.waveform_batch_size], cfg.frequencies,
    batch_size=cfg.waveform_batch_size, _precompiled=precompiled,
)

# Timed: generate 10 batches
n_bench = len(test_params)
t0 = time.time()
wf = generate_waveforms_batch(
    predict_fn, test_params, cfg.frequencies,
    batch_size=cfg.waveform_batch_size, _precompiled=precompiled,
)
elapsed = time.time() - t0

print(f"Generated {n_bench} waveforms × {cfg.n_freq} freq points")
print(f"  Total time    : {elapsed:.2f}s")
print(f"  Per waveform  : {elapsed / n_bench * 1000:.1f} ms")
print(f"  Throughput    : {n_bench / elapsed:.1f} waveforms/s")
print(f"  Waveform shape: {wf.shape}")
print(f"  Waveform dtype: {wf.dtype}")
print(f"  Memory per wf : {wf[0].nbytes / 1e6:.1f} MB")

del wf, test_params
import gc; gc.collect()

## 6. Build the ROQ basis

This is the main computation. The build has three phases per basis kind:

1. **Pre-selection (greedy)** — scan corner + random waveforms, pick the worst-represented one at each step
2. **Enrichment** — stream larger training sets, add any waveform above tolerance
3. **EIM** — select empirical interpolation nodes from the basis

Each phase saves a **checkpoint** to disk. If the Colab runtime
disconnects, re-run this cell — it will **resume** from the last
completed phase automatically.

**Expected timing (T4 GPU):**
- Linear pre-selection: ~30–60 min
- Linear enrichment: ~1–3 hours
- Quadratic: faster (smaller basis)

💡 **Tip:** Mount Google Drive (cell above) to persist checkpoints
across sessions.

In [ ]:
from roq_builder_jax import build_roq_basis
import gc

print("=" * 65)
print("  Building LINEAR ROQ basis")
print("=" * 65)
print(f"  Resume mode: ON (will skip completed phases)")
print()

t0 = time.time()
results_lin = build_roq_basis(cfg, kind="linear", resume=True)
t_lin = time.time() - t0

print(f"\n{'─' * 50}")
print(f"LINEAR basis built in {t_lin:.1f}s ({t_lin/60:.1f} min)")
print(f"  Basis size    : {len(results_lin['basis'])}")
print(f"  Nodes         : {len(results_lin['nodes'])}")
print(f"  Compression   : {cfg.n_freq / len(results_lin['nodes']):.0f}×")

In [ ]:
# Free linear waveform data before quadratic build
del results_lin["basis"]  # keep nodes/interpolant for later
gc.collect()

print("=" * 65)
print("  Building QUADRATIC ROQ basis")
print("=" * 65)
print(f"  Resume mode: ON")
print()

t0 = time.time()
results_qua = build_roq_basis(cfg, kind="quadratic", resume=True)
t_qua = time.time() - t0

print(f"\n{'─' * 50}")
print(f"QUADRATIC basis built in {t_qua:.1f}s ({t_qua/60:.1f} min)")
print(f"  Basis size    : {len(results_qua['basis'])}")
print(f"  Nodes         : {len(results_qua['nodes'])}")
print(f"  Compression   : {cfg.n_freq / len(results_qua['nodes']):.0f}×")

## 7. Validate the basis

Generate random unseen waveforms and check that the ROQ reconstruction
error is below the tolerance.

In [ ]:
from roq_builder_jax import validate_basis
from pathlib import Path

# Reload results from disk (in case we resumed)
out_lin = Path(cfg.output_dir) / "ROQ_data" / "linear"
out_qua = Path(cfg.output_dir) / "ROQ_data" / "quadratic"

results_lin_disk = {
    "basis": np.load(out_lin / "basis_linear.npy"),
    "nodes": np.load(out_lin / "empirical_nodes_linear.npy"),
    "interpolant": np.load(out_lin / "basis_interpolant_linear.npy"),
    "frequencies": cfg.frequencies,
}

# Re-load predictor and warm up for validation
predict_fn_val = _load_predictor(cfg.model_path)
precompiled_val = warmup_jit(
    predict_fn_val, cfg.frequencies, cfg.waveform_batch_size, verbose=0,
)

print("Validating LINEAR basis...")
errors_lin, lin_ok = validate_basis(
    results_lin_disk, predict_fn_val, cfg,
    n_test=200, quadratic=False, precompiled=precompiled_val,
)

if out_qua.exists() and (out_qua / "basis_quadratic.npy").exists():
    results_qua_disk = {
        "basis": np.load(out_qua / "basis_quadratic.npy"),
        "nodes": np.load(out_qua / "empirical_nodes_quadratic.npy"),
        "interpolant": np.load(out_qua / "basis_interpolant_quadratic.npy"),
        "frequencies": cfg.frequencies,
    }
    print("\nValidating QUADRATIC basis...")
    errors_qua, qua_ok = validate_basis(
        results_qua_disk, predict_fn_val, cfg,
        n_test=200, quadratic=True, precompiled=precompiled_val,
    )
else:
    print("\n(Quadratic basis not yet built — skipping validation)")

## 8. Summary and output files

In [ ]:
import json
from pathlib import Path

out_dir = Path(cfg.output_dir)

print("=" * 65)
print("  ROQ BUILD OUTPUT FILES")
print("=" * 65)

for kind in ["linear", "quadratic"]:
    d = out_dir / "ROQ_data" / kind
    if not d.exists():
        print(f"\n  {kind.upper()}: not built yet")
        continue
    print(f"\n  {kind.upper()} basis:")
    for f in sorted(d.iterdir()):
        size = f.stat().st_size
        if size > 1e6:
            print(f"    {f.name:50s}  {size/1e6:.1f} MB")
        else:
            print(f"    {f.name:50s}  {size/1e3:.1f} KB")

# Print status markers
print(f"\n{'─' * 50}")
print("  Checkpoint status:")
for kind in ["linear", "quadratic"]:
    d = out_dir / "ROQ_data" / kind
    status_file = d / f"_status_{kind}.json"
    if status_file.exists():
        with open(status_file) as f:
            status = json.load(f)
        phases = list(status.keys())
        print(f"    {kind}: {', '.join(phases)} ✓")
    else:
        print(f"    {kind}: no checkpoints")

print(f"\n  Output directory: {out_dir.resolve()}")
print("\n  Files needed for PE (small, can be downloaded):")
print("    - empirical_frequencies_{linear,quadratic}.npy")
print("    - empirical_nodes_{linear,quadratic}.npy")
print("    - basis_interpolant_{linear,quadratic}.npy")

## 9. (Optional) Save results to Google Drive

If you mounted Google Drive above, the results are already saved there.
Otherwise, you can copy them now:

In [ ]:
# Uncomment to mount Google Drive and copy results
# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil
# dest = '/content/drive/MyDrive/roq_basis_mlgw_bns_jax'
# shutil.copytree(cfg.output_dir, dest, dirs_exist_ok=True)
# print(f"✓ Results copied to {dest}")

## 10. (Optional) Download key files

Download just the small files needed for ROQ-accelerated PE:

In [ ]:
# Uncomment to download files directly from Colab
# from google.colab import files
# for kind in ["linear", "quadratic"]:
#     d = Path(cfg.output_dir) / "ROQ_data" / kind
#     for name in ["empirical_frequencies", "empirical_nodes", "basis_interpolant"]:
#         fpath = d / f"{name}_{kind}.npy"
#         if fpath.exists():
#             files.download(str(fpath))
#             print(f"Downloaded {fpath.name}")